# Evaluation
## Prepare the environment

In [1]:
LOCAL = True

In [2]:
import torch
import matplotlib
matplotlib.use("Agg")

import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from dataset import MotionDataset
from sample import denormalize_samples, generate
from metrics import compute_fmd, compute_mpjpe, compute_mpjpe_v2, compute_sample_variance
from utils import animate_skeleton_3d, JOINT_CONNECTIONS

In [3]:
if LOCAL:
    TRAIN_DATA_PATH = Path("data/train.npz")
    TEST_DATA_PATH = Path("data/test.npz")
    NORM_STATS_PATH = Path("data/norm_stats.npy")
    BEST_MODEL_PATH = Path("best_model/best_model.pt")

    RESULTS_PATH_TRAIN = Path("results/evaluation/train")
    RESULTS_PATH_TRAIN.mkdir(parents=True, exist_ok=True)

    RESULTS_PATH_TEST = Path("results/evaluation/test")
    RESULTS_PATH_TEST.mkdir(parents=True, exist_ok=True)
else:
    raise NotImplementedError("This code is meant to be run locally.")

print(f"Train data: {TRAIN_DATA_PATH} (exists: {TRAIN_DATA_PATH.exists()})")
print(f"Test data: {TEST_DATA_PATH} (exists: {TEST_DATA_PATH.exists()})")
print(f"Norm stats: {NORM_STATS_PATH} (exists: {NORM_STATS_PATH.exists()})")
print(f"Model: {BEST_MODEL_PATH} (exists: {BEST_MODEL_PATH.exists()})")
print(f"Train results dir: {RESULTS_PATH_TRAIN} (exists: {RESULTS_PATH_TRAIN.exists()})")
print(f"Test results dir: {RESULTS_PATH_TEST} (exists: {RESULTS_PATH_TEST.exists()})")

Train data: data/train.npz (exists: True)
Test data: data/test.npz (exists: True)
Norm stats: data/norm_stats.npy (exists: True)
Model: best_model/best_model.pt (exists: True)
Train results dir: results/evaluation/train (exists: True)
Test results dir: results/evaluation/test (exists: True)


In [4]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU


## Train set
### Load the data

In [5]:
train_dataset = MotionDataset(str(TRAIN_DATA_PATH))
seq, label = train_dataset[0]

print(f"Train samples: {len(train_dataset)}")
print(f"Sequence shape: {seq.shape}")
print(f"Num classes: {int(train_dataset.labels.max()) + 1}")

train_sequences = train_dataset.sequences.numpy().astype(np.float32)
labels_train = train_dataset.labels.numpy().astype(np.int64)
n_frames = train_sequences.shape[1]
n_joints = train_sequences.shape[2]
print(f"Train set: {train_sequences.shape} labels: {np.bincount(labels_train)}")

Train samples: 1260
Sequence shape: torch.Size([48, 15, 3])
Num classes: 2
Train set: (1260, 48, 15, 3) labels: [623 637]


In [6]:
seq_tensor = denormalize_samples(torch.tensor(train_sequences), str(NORM_STATS_PATH))
train_sequences = seq_tensor.numpy()
stats = np.load(NORM_STATS_PATH)
print(f"Denormalized train data (mean={stats[0]:.4f}, std={stats[1]:.4f})")

Denormalized train data (mean=-0.6862, std=10.6511)


In [7]:
real_train_walk_mask = labels_train == 0
real_train_walk = train_sequences[real_train_walk_mask]
print(f"Train real walk samples : {len(real_train_walk)}")

real_train_jump_mask = labels_train == 1
real_train_jump = train_sequences[real_train_jump_mask]
print(f"Train real jump samples : {len(real_train_jump)}")

Train real walk samples : 623
Train real jump samples : 637


### Generate predictions

In [8]:
walk_generated_train = generate(
    checkpoint_path=str(BEST_MODEL_PATH),
    class_label=0,
    n_samples=len(real_train_walk),
    guidance_scale=3.0,
    n_joints=15,
    n_frames=48,
    d_model=384,
    nhead=6,
    num_layers=6,
    num_classes=2,
    dropout=0.1,
    timesteps=1000,
    device=DEVICE,
)

walk_gen_train = denormalize_samples(walk_generated_train, str(NORM_STATS_PATH)).numpy()
print(f"Generated walk samples (denormalised) (shape: {walk_gen_train.shape})")

Generated walk samples (denormalised) (shape: (623, 48, 15, 3))


In [9]:
jump_generated_train = generate(
    checkpoint_path=str(BEST_MODEL_PATH),
    class_label=1,
    n_samples=len(real_train_jump),
    guidance_scale=3.0,
    n_joints=15,
    n_frames=48,
    d_model=384,
    nhead=6,
    num_layers=6,
    num_classes=2,
    dropout=0.1,
    timesteps=1000,
    device=DEVICE,
)

jump_gen_train = denormalize_samples(jump_generated_train, str(NORM_STATS_PATH)).numpy()
print(f"Generated jump samples (denormalised) (shape: {jump_gen_train.shape})")

Generated jump samples (denormalised) (shape: (637, 48, 15, 3))


### Calculate metrics

In [10]:
walk_fmd = compute_fmd(real_train_walk, walk_gen_train)
jump_fmd = compute_fmd(real_train_jump, jump_gen_train)

print(f"FMD (Fréchet Motion Distance)")
print("  ↳ Lower = generated distribution closer to real.")
print(f"  Walk: {walk_fmd:.4f}")
print(f"  Jump: {jump_fmd:.4f}")

FMD (Fréchet Motion Distance)
  ↳ Lower = generated distribution closer to real.
  Walk: 31.0329
  Jump: 139.7607


In [11]:
mpjpe_nn_walk = compute_mpjpe(real_train_walk, walk_gen_train)
mpjpe_mean_walk = compute_mpjpe_v2(real_train_walk, walk_gen_train)

mpjpe_nn_jump = compute_mpjpe(real_train_jump, jump_gen_train)
mpjpe_mean_jump = compute_mpjpe_v2(real_train_jump, jump_gen_train)

print(f"MPJPE (nearest-neighbour pairing)")
print("  ↳ Lower = generated joints closer to real joints.")
print(f"  Walk: {mpjpe_nn_walk:.4f}")
print(f"  Jump: {mpjpe_nn_jump:.4f}")
print()
print(f"MPJPE (vs mean real pose)")
print("  ↳ Lower = generated joints closer to real joints.")
print(f"  Walk: {mpjpe_mean_walk:.4f}")
print(f"  Jump: {mpjpe_mean_jump:.4f}")

MPJPE (nearest-neighbour pairing)
  ↳ Lower = generated joints closer to real joints.
  Walk: 2.7094
  Jump: 2.0850

MPJPE (vs mean real pose)
  ↳ Lower = generated joints closer to real joints.
  Walk: 19.6607
  Jump: 6.2229


In [12]:
var_walk = compute_sample_variance(walk_gen_train)
var_jump = compute_sample_variance(jump_gen_train)

print(f"Sample Diversity:")
print("  ↳ Higher = more creative / diverse outputs.")
print()
print(f"  Mean pairwise distance (feat)")
print(f"  Walk: {var_walk['mean_pairwise_dist']:.4f}")
print(f"  Jump: {var_jump['mean_pairwise_dist']:.4f}")
print()
print(f"  Joint position std (across samp)")
print(f"  Walk: {var_walk['joint_position_std']:.4f}")
print(f"  Jump: {var_jump['joint_position_std']:.4f}")
print()
print(f"  Velocity std (across samp)")
print(f"  Walk: {var_walk['velocity_std']:.4f}")
print(f"  Jump: {var_jump['velocity_std']:.4f}")

Sample Diversity:
  ↳ Higher = more creative / diverse outputs.

  Mean pairwise distance (feat)
  Walk: 51.8356
  Jump: 22.8551

  Joint position std (across samp)
  Walk: 9.5011
  Jump: 3.4029

  Velocity std (across samp)
  Walk: 0.9630
  Jump: 0.4805


### Summary

In [13]:
results_train = {
    "walk": {
        "fmd": walk_fmd,
        "mpjpe_nn": mpjpe_nn_walk,
        "mpjpe_vs_mean": mpjpe_mean_walk,
        "mean_pairwise_dist": var_walk["mean_pairwise_dist"],
        "joint_position_std": var_walk["joint_position_std"],
        "velocity_std": var_walk["velocity_std"],
        "n_real": int(len(real_train_walk)),
    },
    "jump": {
        "fmd": jump_fmd,
        "mpjpe_nn": mpjpe_nn_jump,
        "mpjpe_vs_mean": mpjpe_mean_jump,
        "mean_pairwise_dist": var_jump["mean_pairwise_dist"],
        "joint_position_std": var_jump["joint_position_std"],
        "velocity_std": var_jump["velocity_std"],
        "n_real": int(len(real_train_jump)),
    }
}

In [14]:
print(f"\n\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")

header = f"{'Metric':<35}"

for cls in results_train:
    header += f"  {cls.upper():>10}"

print(header)
print("-" * len(header))

metric_keys = [
    ("FMD", "fmd"),
    ("MPJPE (NN)", "mpjpe_nn"),
    ("MPJPE (vs mean real)", "mpjpe_vs_mean"),
    ("Diversity – pairwise", "mean_pairwise_dist"),
    ("Diversity – joint std", "joint_position_std"),
    ("Diversity – vel std", "velocity_std"),
]

for label, key in metric_keys:
    row = f"{label:<35}"

    for cls in results_train:
        row += f"  {results_train[cls][key]:>10.4f}"

    print(row)



SUMMARY
Metric                                     WALK        JUMP
-----------------------------------------------------------
FMD                                     31.0329    139.7607
MPJPE (NN)                               2.7094      2.0850
MPJPE (vs mean real)                    19.6607      6.2229
Diversity – pairwise                    51.8356     22.8551
Diversity – joint std                    9.5011      3.4029
Diversity – vel std                      0.9630      0.4805


In [15]:
print("\n\n" + "=" * 60)
print("MARKDOWN TABLE")
print("=" * 60)

md  = "| **Ruch** | **FMD** | **MPJPE** | **Var** |\n"
md += "|:--------:|--------:|----------:|--------:|\n"

for cls_name, res in results_train.items():
    md += (
        f"| *{cls_name}* "
        f"| {res['fmd']:.4f} "
        f"| {res['mpjpe_nn']:.4f} "
        f"| {res['joint_position_std']:.4f} |\n"
    )

print(md)



MARKDOWN TABLE
| **Ruch** | **FMD** | **MPJPE** | **Var** |
|:--------:|--------:|----------:|--------:|
| *walk* | 31.0329 | 2.7094 | 9.5011 |
| *jump* | 139.7607 | 2.0850 | 3.4029 |



In [16]:
print("-" * 60)
print("EXTENDED TABLE (all sub-metrics)\n")

ext  = "| **Ruch** | **FMD** | **MPJPE (NN)** | **MPJPE (vs mean)** | **Div – pairwise** | **Div – joint std** | **Div – vel std** |\n"
ext += "|:--------:|--------:|---------------:|--------------------:|-------------------:|--------------------:|------------------:|\n"

for cls_name, res in results_train.items():
    ext += (
        f"| *{cls_name}* "
        f"| {res['fmd']:.4f} "
        f"| {res['mpjpe_nn']:.4f} "
        f"| {res['mpjpe_vs_mean']:.4f} "
        f"| {res['mean_pairwise_dist']:.4f} "
        f"| {res['joint_position_std']:.4f} "
        f"| {res['velocity_std']:.4f} |\n"
    )

print(ext)

------------------------------------------------------------
EXTENDED TABLE (all sub-metrics)

| **Ruch** | **FMD** | **MPJPE (NN)** | **MPJPE (vs mean)** | **Div – pairwise** | **Div – joint std** | **Div – vel std** |
|:--------:|--------:|---------------:|--------------------:|-------------------:|--------------------:|------------------:|
| *walk* | 31.0329 | 2.7094 | 19.6607 | 51.8356 | 9.5011 | 0.9630 |
| *jump* | 139.7607 | 2.0850 | 6.2229 | 22.8551 | 3.4029 | 0.4805 |



### Visualization

In [17]:
def visualize_nearest_neighbours(
    real: np.ndarray,
    generated: np.ndarray,
    cls_name: str,
    n_viz: int = 10,
    fps: int = 8,
    save_dir: str = "eval_viz",
    seed: int = 0,
) -> None:

    rng = np.random.default_rng(seed)
    N_g = generated.shape[0]
    N_r = real.shape[0]
    n_viz = min(n_viz, N_g)

    gen_flat  = generated.reshape(N_g, -1)
    real_flat = real.reshape(N_r, -1)

    gen_indices = rng.choice(N_g, size=n_viz, replace=False)

    save_dir = Path(save_dir)
    save_dir.mkdir(parents=True, exist_ok=True)

    for pair_idx, gi in enumerate(gen_indices, start=1):
        dists  = np.linalg.norm(real_flat - gen_flat[gi], axis=1)
        ri     = int(np.argmin(dists))
        dist   = float(dists[ri])

        gen_path  = str(save_dir / f"{cls_name}_pair{pair_idx:02d}_generated.gif")
        real_path = str(save_dir / f"{cls_name}_pair{pair_idx:02d}_real_nn.gif")

        animate_skeleton_3d(generated[gi], output_filename=gen_path, fps=fps, show=False)
        animate_skeleton_3d(real[ri], output_filename=real_path, fps=fps, show=False)

        print(f"  pair {pair_idx:02d}  gen#{gi} ↔ real#{ri}  (d={dist:.3f})")
        print(f"    generated → {gen_path}")
        print(f"    real      → {real_path}")

In [18]:
visualize_nearest_neighbours(
    real=real_train_walk, 
    generated=walk_gen_train,
    cls_name="walk",
    save_dir=str(RESULTS_PATH_TRAIN)
)

  pair 01  gen#522 ↔ real#608  (d=57.762)
    generated → results/evaluation/train/walk_pair01_generated.gif
    real      → results/evaluation/train/walk_pair01_real_nn.gif
  pair 02  gen#506 ↔ real#105  (d=182.087)
    generated → results/evaluation/train/walk_pair02_generated.gif
    real      → results/evaluation/train/walk_pair02_real_nn.gif
  pair 03  gen#391 ↔ real#123  (d=105.191)
    generated → results/evaluation/train/walk_pair03_generated.gif
    real      → results/evaluation/train/walk_pair03_real_nn.gif
  pair 04  gen#314 ↔ real#552  (d=55.631)
    generated → results/evaluation/train/walk_pair04_generated.gif
    real      → results/evaluation/train/walk_pair04_real_nn.gif
  pair 05  gen#166 ↔ real#529  (d=65.650)
    generated → results/evaluation/train/walk_pair05_generated.gif
    real      → results/evaluation/train/walk_pair05_real_nn.gif
  pair 06  gen#25 ↔ real#371  (d=59.564)
    generated → results/evaluation/train/walk_pair06_generated.gif
    real      → resu

In [19]:
visualize_nearest_neighbours(
    real=real_train_jump, 
    generated=jump_gen_train,
    cls_name="jump",
    save_dir=str(RESULTS_PATH_TRAIN)
)

  pair 01  gen#534 ↔ real#604  (d=92.293)
    generated → results/evaluation/train/jump_pair01_generated.gif
    real      → results/evaluation/train/jump_pair01_real_nn.gif
  pair 02  gen#518 ↔ real#604  (d=52.075)
    generated → results/evaluation/train/jump_pair02_generated.gif
    real      → results/evaluation/train/jump_pair02_real_nn.gif
  pair 03  gen#400 ↔ real#630  (d=68.708)
    generated → results/evaluation/train/jump_pair03_generated.gif
    real      → results/evaluation/train/jump_pair03_real_nn.gif
  pair 04  gen#322 ↔ real#584  (d=51.093)
    generated → results/evaluation/train/jump_pair04_generated.gif
    real      → results/evaluation/train/jump_pair04_real_nn.gif
  pair 05  gen#170 ↔ real#264  (d=91.503)
    generated → results/evaluation/train/jump_pair05_generated.gif
    real      → results/evaluation/train/jump_pair05_real_nn.gif
  pair 06  gen#25 ↔ real#446  (d=71.613)
    generated → results/evaluation/train/jump_pair06_generated.gif
    real      → result

## Test set
### Load the data

In [20]:
test_dataset = MotionDataset(str(TEST_DATA_PATH))
seq, label = test_dataset[0]

print(f"Test samples: {len(test_dataset)}")
print(f"Sequence shape: {seq.shape}")
print(f"Num classes: {int(test_dataset.labels.max()) + 1}")

test_sequences = test_dataset.sequences.numpy().astype(np.float32)
labels_test = test_dataset.labels.numpy().astype(np.int64)
n_frames = test_sequences.shape[1]
n_joints = test_sequences.shape[2]
print(f"Test set: {test_sequences.shape} labels: {np.bincount(labels_test)}")

Test samples: 35
Sequence shape: torch.Size([48, 15, 3])
Num classes: 2
Test set: (35, 48, 15, 3) labels: [23 12]


In [21]:
seq_tensor = denormalize_samples(torch.tensor(test_sequences), str(NORM_STATS_PATH))
test_sequences = seq_tensor.numpy()
stats = np.load(NORM_STATS_PATH)
print(f"Denormalized test data (mean={stats[0]:.4f}, std={stats[1]:.4f})")

Denormalized test data (mean=-0.6862, std=10.6511)


In [22]:
real_test_walk_mask = labels_test == 0
real_test_walk = test_sequences[real_test_walk_mask]
print(f"Test real walk samples : {len(real_test_walk)}")

real_test_jump_mask = labels_test == 1
real_test_jump = test_sequences[real_test_jump_mask]
print(f"Test real jump samples : {len(real_test_jump)}")

Test real walk samples : 23
Test real jump samples : 12


### Generate predictions

In [23]:
walk_generated_test = generate(
    checkpoint_path=str(BEST_MODEL_PATH),
    class_label=0,
    n_samples=len(real_test_walk),
    guidance_scale=3.0,
    n_joints=15,
    n_frames=48,
    d_model=384,
    nhead=6,
    num_layers=6,
    num_classes=2,
    dropout=0.1,
    timesteps=1000,
    device=DEVICE,
)

walk_gen_test = denormalize_samples(walk_generated_test, str(NORM_STATS_PATH)).numpy()
print(f"Generated walk samples (denormalised) (shape: {walk_gen_test.shape})")

Generated walk samples (denormalised) (shape: (23, 48, 15, 3))


In [24]:
jump_generated_test = generate(
    checkpoint_path=str(BEST_MODEL_PATH),
    class_label=1,
    n_samples=len(real_test_jump),
    guidance_scale=3.0,
    n_joints=15,
    n_frames=48,
    d_model=384,
    nhead=6,
    num_layers=6,
    num_classes=2,
    dropout=0.1,
    timesteps=1000,
    device=DEVICE,
)

jump_gen_test = denormalize_samples(jump_generated_test, str(NORM_STATS_PATH)).numpy()
print(f"Generated jump samples (denormalised) (shape: {jump_gen_test.shape})")

Generated jump samples (denormalised) (shape: (12, 48, 15, 3))


### Calculate metrics

In [25]:
walk_fmd = compute_fmd(real_test_walk, walk_gen_test)
jump_fmd = compute_fmd(real_test_jump, jump_gen_test)

print(f"FMD (Fréchet Motion Distance)")
print("  ↳ Lower = generated distribution closer to real.")
print(f"  Walk: {walk_fmd:.4f}")
print(f"  Jump: {jump_fmd:.4f}")

FMD (Fréchet Motion Distance)
  ↳ Lower = generated distribution closer to real.
  Walk: 2596.0407
  Jump: 882.1705


In [26]:
mpjpe_nn_walk = compute_mpjpe(real_test_walk, walk_gen_test)
mpjpe_mean_walk = compute_mpjpe_v2(real_test_walk, walk_gen_test)

mpjpe_nn_jump = compute_mpjpe(real_test_jump, jump_gen_test)
mpjpe_mean_jump = compute_mpjpe_v2(real_test_jump, jump_gen_test)

print(f"MPJPE (nearest-neighbour pairing)")
print("  ↳ Lower = generated joints closer to real joints.")
print(f"  Walk: {mpjpe_nn_walk:.4f}")
print(f"  Jump: {mpjpe_nn_jump:.4f}")
print()
print(f"MPJPE (vs mean real pose)")
print("  ↳ Lower = generated joints closer to real joints.")
print(f"  Walk: {mpjpe_mean_walk:.4f}")
print(f"  Jump: {mpjpe_mean_jump:.4f}")

MPJPE (nearest-neighbour pairing)
  ↳ Lower = generated joints closer to real joints.
  Walk: 10.5151
  Jump: 7.1853

MPJPE (vs mean real pose)
  ↳ Lower = generated joints closer to real joints.
  Walk: 19.2417
  Jump: 7.7266


In [27]:
var_walk = compute_sample_variance(walk_gen_test)
var_jump = compute_sample_variance(jump_gen_test)

print(f"Sample Diversity:")
print("  ↳ Higher = more creative / diverse outputs.")
print()
print(f"  Mean pairwise distance (feat)")
print(f"  Walk: {var_walk['mean_pairwise_dist']:.4f}")
print(f"  Jump: {var_jump['mean_pairwise_dist']:.4f}")
print()
print(f"  Joint position std (across samp)")
print(f"  Walk: {var_walk['joint_position_std']:.4f}")
print(f"  Jump: {var_jump['joint_position_std']:.4f}")
print()
print(f"  Velocity std (across samp)")
print(f"  Walk: {var_walk['velocity_std']:.4f}")
print(f"  Jump: {var_jump['velocity_std']:.4f}")

Sample Diversity:
  ↳ Higher = more creative / diverse outputs.

  Mean pairwise distance (feat)
  Walk: 42.3766
  Jump: 24.0175

  Joint position std (across samp)
  Walk: 8.6794
  Jump: 3.5913

  Velocity std (across samp)
  Walk: 0.8457
  Jump: 0.4572


### Summary

In [28]:
results_test = {
    "walk": {
        "fmd": walk_fmd,
        "mpjpe_nn": mpjpe_nn_walk,
        "mpjpe_vs_mean": mpjpe_mean_walk,
        "mean_pairwise_dist": var_walk["mean_pairwise_dist"],
        "joint_position_std": var_walk["joint_position_std"],
        "velocity_std": var_walk["velocity_std"],
        "n_real": int(len(real_train_walk)),
    },
    "jump": {
        "fmd": jump_fmd,
        "mpjpe_nn": mpjpe_nn_jump,
        "mpjpe_vs_mean": mpjpe_mean_jump,
        "mean_pairwise_dist": var_jump["mean_pairwise_dist"],
        "joint_position_std": var_jump["joint_position_std"],
        "velocity_std": var_jump["velocity_std"],
        "n_real": int(len(real_train_jump)),
    }
}

In [29]:
print(f"\n\n{'='*60}")
print("SUMMARY")
print(f"{'='*60}")

header = f"{'Metric':<35}"

for cls in results_test:
    header += f"  {cls.upper():>10}"

print(header)
print("-" * len(header))

metric_keys = [
    ("FMD", "fmd"),
    ("MPJPE (NN)", "mpjpe_nn"),
    ("MPJPE (vs mean real)", "mpjpe_vs_mean"),
    ("Diversity – pairwise", "mean_pairwise_dist"),
    ("Diversity – joint std", "joint_position_std"),
    ("Diversity – vel std", "velocity_std"),
]

for label, key in metric_keys:
    row = f"{label:<35}"

    for cls in results_test:
        row += f"  {results_test[cls][key]:>10.4f}"

    print(row)



SUMMARY
Metric                                     WALK        JUMP
-----------------------------------------------------------
FMD                                   2596.0407    882.1705
MPJPE (NN)                              10.5151      7.1853
MPJPE (vs mean real)                    19.2417      7.7266
Diversity – pairwise                    42.3766     24.0175
Diversity – joint std                    8.6794      3.5913
Diversity – vel std                      0.8457      0.4572


In [30]:
print("\n\n" + "=" * 60)
print("MARKDOWN TABLE")
print("=" * 60)

md  = "| **Ruch** | **FMD** | **MPJPE** | **Var** |\n"
md += "|:--------:|--------:|----------:|--------:|\n"

for cls_name, res in results_test.items():
    md += (
        f"| *{cls_name}* "
        f"| {res['fmd']:.4f} "
        f"| {res['mpjpe_nn']:.4f} "
        f"| {res['joint_position_std']:.4f} |\n"
    )

print(md)



MARKDOWN TABLE
| **Ruch** | **FMD** | **MPJPE** | **Var** |
|:--------:|--------:|----------:|--------:|
| *walk* | 2596.0407 | 10.5151 | 8.6794 |
| *jump* | 882.1705 | 7.1853 | 3.5913 |



In [31]:
print("-" * 60)
print("EXTENDED TABLE (all sub-metrics)\n")

ext  = "| **Ruch** | **FMD** | **MPJPE (NN)** | **MPJPE (vs mean)** | **Div – pairwise** | **Div – joint std** | **Div – vel std** |\n"
ext += "|:--------:|--------:|---------------:|--------------------:|-------------------:|--------------------:|------------------:|\n"

for cls_name, res in results_test.items():
    ext += (
        f"| *{cls_name}* "
        f"| {res['fmd']:.4f} "
        f"| {res['mpjpe_nn']:.4f} "
        f"| {res['mpjpe_vs_mean']:.4f} "
        f"| {res['mean_pairwise_dist']:.4f} "
        f"| {res['joint_position_std']:.4f} "
        f"| {res['velocity_std']:.4f} |\n"
    )

print(ext)

------------------------------------------------------------
EXTENDED TABLE (all sub-metrics)

| **Ruch** | **FMD** | **MPJPE (NN)** | **MPJPE (vs mean)** | **Div – pairwise** | **Div – joint std** | **Div – vel std** |
|:--------:|--------:|---------------:|--------------------:|-------------------:|--------------------:|------------------:|
| *walk* | 2596.0407 | 10.5151 | 19.2417 | 42.3766 | 8.6794 | 0.8457 |
| *jump* | 882.1705 | 7.1853 | 7.7266 | 24.0175 | 3.5913 | 0.4572 |



### Visualization

In [32]:
visualize_nearest_neighbours(
    real=real_test_walk, 
    generated=walk_gen_test,
    cls_name="walk",
    save_dir=str(RESULTS_PATH_TEST)
)

  pair 01  gen#11 ↔ real#11  (d=331.319)
    generated → results/evaluation/test/walk_pair01_generated.gif
    real      → results/evaluation/test/walk_pair01_real_nn.gif
  pair 02  gen#18 ↔ real#9  (d=341.592)
    generated → results/evaluation/test/walk_pair02_generated.gif
    real      → results/evaluation/test/walk_pair02_real_nn.gif
  pair 03  gen#9 ↔ real#13  (d=254.594)
    generated → results/evaluation/test/walk_pair03_generated.gif
    real      → results/evaluation/test/walk_pair03_real_nn.gif
  pair 04  gen#8 ↔ real#13  (d=465.449)
    generated → results/evaluation/test/walk_pair04_generated.gif
    real      → results/evaluation/test/walk_pair04_real_nn.gif
  pair 05  gen#4 ↔ real#14  (d=215.348)
    generated → results/evaluation/test/walk_pair05_generated.gif
    real      → results/evaluation/test/walk_pair05_real_nn.gif
  pair 06  gen#0 ↔ real#14  (d=183.521)
    generated → results/evaluation/test/walk_pair06_generated.gif
    real      → results/evaluation/test/wal

In [33]:
visualize_nearest_neighbours(
    real=real_test_jump, 
    generated=jump_gen_test,
    cls_name="jump",
    save_dir=str(RESULTS_PATH_TEST)
)

  pair 01  gen#2 ↔ real#7  (d=201.303)
    generated → results/evaluation/test/jump_pair01_generated.gif
    real      → results/evaluation/test/jump_pair01_real_nn.gif
  pair 02  gen#11 ↔ real#1  (d=241.335)
    generated → results/evaluation/test/jump_pair02_generated.gif
    real      → results/evaluation/test/jump_pair02_real_nn.gif
  pair 03  gen#3 ↔ real#4  (d=138.560)
    generated → results/evaluation/test/jump_pair03_generated.gif
    real      → results/evaluation/test/jump_pair03_real_nn.gif
  pair 04  gen#4 ↔ real#11  (d=103.623)
    generated → results/evaluation/test/jump_pair04_generated.gif
    real      → results/evaluation/test/jump_pair04_real_nn.gif
  pair 05  gen#1 ↔ real#7  (d=220.702)
    generated → results/evaluation/test/jump_pair05_generated.gif
    real      → results/evaluation/test/jump_pair05_real_nn.gif
  pair 06  gen#0 ↔ real#1  (d=151.040)
    generated → results/evaluation/test/jump_pair06_generated.gif
    real      → results/evaluation/test/jump_pai